In [1]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import pandas as pd

In [4]:
df_multi = pd.read_csv("../../data/ml_multiclass_data.csv") 

In [42]:
df_multi.head()

,attack_type,flow_id,cwt_scale_0,cwt_scale_1,cwt_scale_2,cwt_scale_3,cwt_scale_4,cwt_scale_5,cwt_scale_6,cwt_scale_7,...,cwt_scale_126,FLOW_DURATION_MILLISECONDS,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV
0,Benign,924483,-0.743641,3.605484,3.560675,7.368320,4.713295,1.789026,0.193087,0.210379,...,0.044264,584,0,196,14,33,0,276,14,42
1,Benign,1991972,-0.551598,3.172561,3.802309,7.592432,5.234969,2.414897,0.476657,-0.646720,...,0.000130,1151,0,999,88,264,0,1000,67,234
2,Benign,1595752,-0.664497,0.663171,3.162886,7.477042,5.324979,2.265312,0.201699,-0.003630,...,-0.322978,36,0,19,6,8,0,23,7,10
3,Benign,1438936,-0.551654,4.884650,3.997877,7.146777,4.461126,1.826211,0.172284,-0.716152,...,0.241846,6,0,1,0,0,0,1,0,0
4,Benign,598588,-0.551654,4.884650,3.997877,7.146777,4.461126,1.826211,0.172284,-0.716152,...,0.241846,19,0,3,0,0,0,3,0,0


In [5]:
print(df_multi.columns)
print(df_multi['attack_type'].value_counts())

Index(['attack_type', 'flow_id', 'cwt_scale_0', 'cwt_scale_1', 'cwt_scale_2',
       'cwt_scale_3', 'cwt_scale_4', 'cwt_scale_5', 'cwt_scale_6',
       'cwt_scale_7',
       ...
       'cwt_scale_126', 'FLOW_DURATION_MILLISECONDS', 'SRC_TO_DST_IAT_MIN',
       'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_STDDEV',
       'DST_TO_SRC_IAT_MIN', 'DST_TO_SRC_IAT_MAX', 'DST_TO_SRC_IAT_AVG',
       'DST_TO_SRC_IAT_STDDEV'],
      dtype='object', length=138)
attack_type
Exploits          4546
DoS               4198
Benign            4131
Fuzzers           3783
Backdoor          3422
Reconnaissance    3275
Shellcode         1585
Analysis          1226
Generic           1207
Worms              136
Name: count, dtype: int64


In [6]:
X = df_multi.filter(like='cwt').values

In [8]:
y = df_multi['attack_type'].values

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

Train shape: (22007, 127) (22007,)
Test shape: (5502, 127) (5502,)


## Random Forest

In [11]:
clf = RandomForestClassifier(n_estimators=100, random_state=42)

In [12]:
clf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [13]:
y_pred = clf.predict(X_test)

In [14]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred))

Accuracy: 0.4054889131225009
                precision    recall  f1-score   support

      Analysis       0.36      0.42      0.39       245
      Backdoor       0.21      0.18      0.20       685
        Benign       0.90      0.93      0.92       826
           DoS       0.30      0.30      0.30       840
      Exploits       0.34      0.34      0.34       909
       Fuzzers       0.31      0.41      0.35       757
       Generic       0.19      0.10      0.13       241
Reconnaissance       0.41      0.41      0.41       655
     Shellcode       0.26      0.21      0.23       317
         Worms       0.00      0.00      0.00        27

      accuracy                           0.41      5502
     macro avg       0.33      0.33      0.33      5502
  weighted avg       0.40      0.41      0.40      5502



In [25]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [26]:
X = df_multi.drop(columns=["attack_type"])
y = df_multi["attack_type"]

In [27]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Logistic Regression

In [20]:
lr = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000,
    random_state=42
)

lr.fit(X_train, y_train)

c:\Users\Lizzie\OneDrive\Documents\TF-ML-NIDS\venv\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'multinomial'


In [21]:
y_pred = lr.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(report)

Accuracy: 0.3993093420574337
                precision    recall  f1-score   support

      Analysis       0.31      0.25      0.28       245
      Backdoor       0.28      0.28      0.28       685
        Benign       0.84      0.92      0.88       826
           DoS       0.32      0.18      0.23       840
      Exploits       0.33      0.39      0.36       909
       Fuzzers       0.27      0.47      0.34       757
       Generic       0.30      0.03      0.05       241
Reconnaissance       0.42      0.36      0.39       655
     Shellcode       0.32      0.26      0.28       317
         Worms       0.00      0.00      0.00        27

      accuracy                           0.40      5502
     macro avg       0.34      0.31      0.31      5502
  weighted avg       0.40      0.40      0.39      5502



c:\Users\Lizzie\OneDrive\Documents\TF-ML-NIDS\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lizzie\OneDrive\Documents\TF-ML-NIDS\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lizzie\OneDrive\Documents\TF-ML-NIDS\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifi

In [22]:
from sklearn.neighbors import KNeighborsClassifier

## K-NN

In [23]:
knn = KNeighborsClassifier(
    n_neighbors=5,
    weights='uniform',
    metric='minkowski'
)

knn.fit(X_train, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [24]:
y_pred = knn.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy}")
print(report)

Accuracy: 0.42584514721919303
                precision    recall  f1-score   support

      Analysis       0.33      0.54      0.41       245
      Backdoor       0.25      0.34      0.29       685
        Benign       0.93      0.94      0.94       826
           DoS       0.32      0.36      0.34       840
      Exploits       0.35      0.28      0.31       909
       Fuzzers       0.33      0.34      0.34       757
       Generic       0.34      0.13      0.19       241
Reconnaissance       0.45      0.40      0.43       655
     Shellcode       0.38      0.28      0.33       317
         Worms       0.00      0.00      0.00        27

      accuracy                           0.43      5502
     macro avg       0.37      0.36      0.36      5502
  weighted avg       0.43      0.43      0.42      5502



In [35]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

## XGBoost

In [ ]:
X = df_multi.drop(columns=['attack_type', 'flow_id'])
y = df_multi['attack_type']

In [38]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Class mapping:")
for class_name, num in zip(le.classes_, range(len(le.classes_))):
    print(f"{num} -> {class_name}")

Class mapping:
0 -> Analysis
1 -> Backdoor
2 -> Benign
3 -> DoS
4 -> Exploits
5 -> Fuzzers
6 -> Generic
7 -> Reconnaissance
8 -> Shellcode
9 -> Worms


In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

In [40]:
model = XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42
)

model.fit(X_train, y_train)

c:\Users\Lizzie\OneDrive\Documents\TF-ML-NIDS\venv\lib\site-packages\xgboost\training.py:199: UserWarning: [17:51:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'multi:softmax'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [41]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.4500181752090149
                precision    recall  f1-score   support

      Analysis       0.42      0.58      0.49       245
      Backdoor       0.29      0.25      0.27       685
        Benign       0.98      0.97      0.97       826
           DoS       0.35      0.32      0.33       840
      Exploits       0.40      0.36      0.38       909
       Fuzzers       0.33      0.51      0.40       757
       Generic       0.22      0.07      0.11       241
Reconnaissance       0.44      0.43      0.43       655
     Shellcode       0.29      0.27      0.28       317
         Worms       0.33      0.04      0.07        27

      accuracy                           0.45      5502
     macro avg       0.41      0.38      0.37      5502
  weighted avg       0.45      0.45      0.44      5502

